# Fine-Tune `google/gemma-3-270m-it` on Banglish & Export to `.litertlm`

This notebook fine-tunes **Gemma 3 270M Instruction-Tuned (`google/gemma-3-270m-it`)** using **LoRA** on ~2,000+ Banglish (Romanized Bengali), Bangla, and English personal finance sentences and exports the merged model to `gemma3-270m-it-q8.litertlm` for on-device inference in `OfflineExpenseAssistant`.

### Pipeline Steps
1. **Install Dependencies** (`transformers`, `peft`, `trl`, `datasets`, `ai-edge-torch`, `mediapipe`)
2. **Generate Banglish Dataset** (`banglish_expense_finetune.jsonl`)
3. **LoRA Fine-Tuning** (`r=16, lora_alpha=32` on attention + MLP layers)
4. **Merge LoRA Adapter & Verify Banglish JSON Output**
5. **Convert & Quantize (INT8) to `gemma3-270m-it-q8.litertlm`**

In [ ]:
# 1. Install Fine-Tuning & LiteRT Export Dependencies
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes huggingface_hub
!pip install -q ai-edge-torch mediapipe

In [ ]:
# 2. Authenticate with Hugging Face (Required for gated google/gemma-3-270m-it)
from huggingface_hub import login
# Paste your Hugging Face token with Gemma 3 license accepted
login()

In [ ]:
# 3. Generate the Banglish Fine-Tuning Dataset
# If running inside the repository, invoke scripts/generate_banglish_finetune_dataset.py directly;
# otherwise run the generator inline.
import os
if os.path.exists("generate_banglish_finetune_dataset.py"):
    !python3 generate_banglish_finetune_dataset.py --output banglish_expense_finetune.jsonl --num-samples 2500
elif os.path.exists("scripts/generate_banglish_finetune_dataset.py"):
    !python3 scripts/generate_banglish_finetune_dataset.py --output banglish_expense_finetune.jsonl --num-samples 2500
else:
    raise FileNotFoundError("Upload scripts/generate_banglish_finetune_dataset.py to your Colab workspace first.")

In [ ]:
# 4. Load Base Model & Configure LoRA Adapter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = "google/gemma-3-270m-it"
OUTPUT_DIR = "./gemma3-270m-banglish-lora"
MERGED_DIR = "./gemma3-270m-banglish-merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    attn_implementation="eager"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

dataset = load_dataset("json", data_files="banglish_expense_finetune.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train samples: {len(dataset['train'])}, Eval samples: {len(dataset['test'])}")

In [ ]:
# 5. Run Supervised Fine-Tuning (SFT)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
# 6. Merge LoRA Weights into Base Model & Verify Banglish Inference
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

test_queries = [
    "riksa vara 60 tk",
    "ponchash tk cha nasta",
    "50 tk sosage ar 320 tk murgi",
    "beton paisi 35000 tk",
    "ei mashe bazar e koto khoroch hoise",
    "amar balance koto"
]

for q in test_queries:
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    outputs = merged_model.generate(**inputs, max_new_tokens=96, do_sample=False)
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Input : {q}\nOutput: {decoded.strip()}\n{'-'*50}")

In [ ]:
# 7. Convert Merged Model to INT8 TFLite & Bundle as .litertlm
import os
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache
from mediapipe.tasks.python.genai import bundler

TFLITE_EXPORT_DIR = "./litert_export"
os.makedirs(TFLITE_EXPORT_DIR, exist_ok=True)

pytorch_model = gemma3.build_model_270m(MERGED_DIR)
export_config = ExportConfig(
    kvcache_layout=kv_cache.KV_LAYOUT_TRANSPOSED,
    mask_as_input=True,
)

converter.convert_to_tflite(
    pytorch_model,
    output_path=TFLITE_EXPORT_DIR,
    output_name_prefix="gemma3-270m-banglish",
    prefill_seq_len=1024,
    kv_cache_max_len=2048,
    quantize="dynamic_int8",
    export_config=export_config,
)

tflite_files = [f for f in os.listdir(TFLITE_EXPORT_DIR) if f.endswith(".tflite")]
assert len(tflite_files) > 0, "No .tflite file generated!"
tflite_path = os.path.join(TFLITE_EXPORT_DIR, tflite_files[0])

LITE_RT_LM_OUTPUT = "gemma3-270m-it-q8.litertlm"
bundle_config = bundler.BundleConfig(
    tflite_model=tflite_path,
    tokenizer_model=os.path.join(MERGED_DIR, "tokenizer.model"),
    start_token="<bos>",
    stop_tokens=["<eos>", "<end_of_turn>"],
    output_filename=LITE_RT_LM_OUTPUT,
    prompt_prefix_user="<start_of_turn>user\n",
    prompt_suffix_user="<end_of_turn>\n<start_of_turn>model\n",
)
bundler.create_bundle(bundle_config)
print(f"Successfully bundled {LITE_RT_LM_OUTPUT}! Copy it to app/src/main/assets/{LITE_RT_LM_OUTPUT}")